In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers scikit-learn tqdm pandas torch

In [ ]:
!pip install -U transformers accelerate

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

DATASET_PATH = "s7_dataset_fixed.csv"
MODEL_NAME = "distilbert/distilbert-base-multilingual-cased"
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 3
SEED = 42

In [ ]:
import csv

input_file = './s7_dataset.csv'
output_file = './s7_dataset_fixed.csv'

def fix_csv_file(input_file, output_file):
  rows_fixed = []

  with open(input_file, 'r', encoding='utf-8-sig', newline='') as infile:
    lines = infile.readlines()

    header = ['message_id', 'text', 'level1_category', 'level2_labels']
    rows_fixed.append(header)

    for i, line in enumerate(lines[1:], start=1):
      line = line.strip()
      if not line:
        continue

      if line.startswith('"') and line.endswith('"'):
        line = line[1:-1]
        line = line.replace('""', '"')

      first_comma = line.find(',')
      if first_comma == -1:
        print(f"Предупреждение: строка {i} пропущена (нет запятой)")
        continue

      message_id = line[:first_comma].strip()
      rest = line[first_comma+1:]

      parts = rest.rsplit(',', 2)

      if len(parts) == 3:
        text = parts[0].strip()
        level1 = parts[1].strip()
        level2 = parts[2].strip()
        rows_fixed.append([message_id, text, level1, level2])
      else:
        print(f"Предупреждение: строка {i} не разобрана корректно")
        continue

    with open(output_file, 'w', encoding='utf-8', newline='') as outfile:
        writer = csv.writer(
          outfile,
          delimiter=',',
          quotechar='"',
          quoting=csv.QUOTE_MINIMAL
          )

        for row in rows_fixed:
          writer.writerow(row)

    print(f"✓ Файл успешно исправлен!")
    print(f"✓ Исходный файл: {input_file}")
    print(f"✓ Исправленный файл: {output_file}")
    print(f"✓ Обработано строк: {len(rows_fixed)} (включая заголовок)")

    return len(rows_fixed)

fix_csv_file(input_file, output_file)

✓ Файл успешно исправлен!
✓ Исходный файл: ./s7_dataset.csv
✓ Исправленный файл: ./s7_dataset_fixed.csv
✓ Обработано строк: 11906 (включая заголовок)


11906

In [ ]:
import numpy as np
from sklearn.metrics import classification_report
import torch
def evaluate_model(model, tokenizer, X_val, Y_val, mlb):

    print("\n--- Оценка модели ---")
    with torch.no_grad():
        all_preds = []
        for i in range(len(X_val)):
            enc = tokenizer(
                X_val[i],
                truncation=True,
                padding='max_length',
                max_length=MAX_LEN,
                return_tensors='pt'
            )
            for k in enc:
                enc[k] = enc[k].to(model.device)
            logits = model(**enc).logits
            sigmoid = torch.sigmoid(logits).cpu().numpy()[0]
            pred_bin = (sigmoid > 0.5).astype(int)
            all_preds.append(pred_bin)
        all_preds = np.vstack(all_preds)

    print(classification_report(Y_val, all_preds, target_names=mlb.classes_, zero_division=0))


In [ ]:
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
from sklearn.metrics import f1_score, classification_report
import joblib

df = pd.read_csv(DATASET_PATH)
df = df[~df['level2_labels'].isna()]

cat_cleanup = {
    "Лояцность и продажи": "Лояльность и продажи",
    "До Вылетa": "До Вылета"
}

df['level1_category'] = df['level1_category'].replace(cat_cleanup)


for l1 in sorted(df['level1_category'].unique()):
    df_sub = df[df['level1_category'] == l1].copy()

    def clean_label(label):
        return label.replace('«', '').replace('»', '').replace('"', '').replace("'", '').replace("”", '').replace("“", '').strip()
    def clean_list(labels):
        return [clean_label(l) for l in labels if clean_label(l)]

    df_sub['level2_list'] = df_sub['level2_labels'].astype(str).str.split(";").apply(clean_list)

    all_labels = [l for labels in df_sub['level2_list'] for l in labels]
    label_counts = Counter(all_labels)
    allowed_labels = set([l for l, cnt in label_counts.items() if cnt >= 2])
    df_sub['level2_list'] = df_sub['level2_list'].apply(lambda labels: [l for l in labels if l in allowed_labels])
    df_sub = df_sub[df_sub['level2_list'].map(len) > 0]
    if len(df_sub) < 30 or len(allowed_labels) < 2:
        print(f"Пропускаю категорию L1 {l1}, слишком мало данных для обучения")
        continue
    print(f"--- Обучаем модель для L1={l1} (строк: {len(df_sub)}, меток: {len(allowed_labels)}) ---")


    mlb = MultiLabelBinarizer()
    Y = mlb.fit_transform(df_sub['level2_list'])
    X_train, X_val, Y_train, Y_val = train_test_split(
        df_sub['text'].tolist(), Y, test_size=0.15, random_state=SEED
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    class L2Dataset(torch.utils.data.Dataset):
        def __init__(self, texts, ys):
            self.texts = texts
            self.labels = ys
        def __len__(self): return len(self.texts)
        def __getitem__(self, i):
            enc = tokenizer(
                self.texts[i],
                padding='max_length',
                truncation=True,
                max_length=MAX_LEN,
                return_tensors='pt'
            )
            item = {k: v.squeeze(0) for k, v in enc.items()}
            item['labels'] = torch.tensor(self.labels[i], dtype=torch.float)
            return item

    train_ds = L2Dataset(X_train, Y_train)
    val_ds = L2Dataset(X_val, Y_val)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=Y.shape[1],
        problem_type="multi_label_classification"
    )
    training_args = TrainingArguments(
        output_dir=f"./l2_models/{l1.replace(' ', '_')}",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        save_strategy="epoch",
        save_total_limit=2,
        seed=SEED,
        fp16=True if torch.cuda.is_available() else False,
        logging_steps=100,
    )
    def compute_metrics(p):
        preds = (torch.sigmoid(torch.tensor(p.predictions)) > 0.5).int().cpu().numpy()
        f1_micro = f1_score(p.label_ids, preds, average='micro', zero_division=0)
        f1_macro = f1_score(p.label_ids, preds, average='macro', zero_division=0)
        return {
            'f1_micro': f1_micro,
            'f1_macro': f1_macro
        }

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        tokenizer=tokenizer
    )

    trainer.train()
    trainer.save_model(f"./l2_models/{l1.replace(' ', '_')}")
    tokenizer.save_pretrained(f"./l2_models/{l1.replace(' ', '_')}")
    joblib.dump(mlb, f"./l2_models/{l1.replace(' ', '_')}/level2_mlb.joblib")

    evaluate_model(model, tokenizer, X_val, Y_val, mlb)

--- Обучаем модель для L1=Вылет Прилет (строк: 1450, меток: 5) ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...


--- Оценка модели ---
              precision    recall  f1-score   support

 Приветствие       0.91      0.94      0.93        34
    Прощание       0.94      0.91      0.93        34

   micro avg       0.93      0.93      0.93        68
   macro avg       0.93      0.93      0.93        68
weighted avg       0.93      0.93      0.93        68
 samples avg       0.93      0.93      0.93        68

--- Обучаем модель для L1=Технические вопросы (строк: 1500, меток: 2) ---


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-717101332.py:94: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss


Step,Training Loss
100,0.111300
200,0.002700



--- Оценка модели ---
                         precision    recall  f1-score   support

            Работа бота       1.00      1.00      1.00       108
Работа сайта приложения       1.00      1.00      1.00       117

              micro avg       1.00      1.00      1.00       225
              macro avg       1.00      1.00      1.00       225
           weighted avg       1.00      1.00      1.00       225
            samples avg       1.00      1.00      1.00       225

